In [1]:
import sys
import os
import json
import requests
from pathlib import Path
from datetime import datetime
import yaml
import pandas as pd

# Add parent directory to path to import EM43 modules
sys.path.append(str(Path.cwd().parent))

# Import logger modules
from logger.register import UserRegistration
from logger.log import EM43Logger, get_logger, test_connection, get_status

# Import EM43 wrapper
from em43_wrapper import EM43Wrapper

print("✅ All imports successful!")
print(f"Current working directory: {Path.cwd()}")

# Check if user is registered
user_config_path = Path.cwd() / "user_config.json"
if not user_config_path.exists():
    print("\n" + "="*60)
    print("⚠️  QUICK SETUP OPTION:")
    print("To test full logging functionality, you can register here:")
    print("  1. Uncomment the registration line below")
    print("  2. Run this cell")
    print("  3. Continue with the rest of the notebook")
    print("Or run: python register.py in terminal")
    print("="*60)
    
    # Quick registration option (uncomment to use)
    # Uncomment the next 3 lines to register a test user:
    # from logger.register import UserRegistration
    # registrar = UserRegistration()
    # registrar.run_registration()
else:
    print("\n✅ User already registered - full functionality available!")


✅ All imports successful!
Current working directory: c:\python-programs-research\EM43\em43_python_refactored\logger

✅ User already registered - full functionality available!


In [2]:
# Check if config file exists
config_path = Path.cwd() / "config_log.yaml"
user_config_path = Path.cwd() / "user_config.json"

print(f"Config file exists: {config_path.exists()}")
print(f"User config exists: {user_config_path.exists()}")

# Load and display config
if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("\n📝 Logger Configuration:")
    print(f"  Enabled: {config['logging']['enabled']}")
    print(f"  API Endpoint: {config['logging']['api_endpoint']}")
    print(f"  Timeout: {config['logging']['timeout']}s")
    print(f"  Model: {config['logging']['default_model']}")
    print(f"  Log Initial: {config['logging']['log_initial']}")
    print(f"  Log Checkpoints: {config['logging']['log_checkpoints']}")
else:
    print("❌ Config file not found!")


Config file exists: True
User config exists: True

📝 Logger Configuration:
  Enabled: True
  API Endpoint: https://jkk4nk8j9g.execute-api.us-east-1.amazonaws.com/api
  Timeout: 20s
  Model: EM43
  Log Initial: True
  Log Checkpoints: True


In [3]:
# Test API health
api_endpoint = "https://jkk4nk8j9g.execute-api.us-east-1.amazonaws.com/api"

try:
    response = requests.get(f"{api_endpoint}/health", timeout=20)
    if response.status_code == 200:
        health_data = response.json()
        print("✅ API Health Check Passed!")
        print(f"  Status: {health_data['status']}")
        print(f"  Timestamp: {health_data['timestamp']}")
        print(f"  Schema Version: {health_data['schema_version']}")
        print(f"  Architecture: {health_data['architecture']}")
    else:
        print(f"❌ API Health Check Failed: {response.status_code}")
        print(f"Response: {response.text}")
except Exception as e:
    print(f"❌ API Health Check Error: {e}")


✅ API Health Check Passed!
  Status: healthy
  Timestamp: 2025-07-11T12:37:38.911355Z
  Schema Version: 3.0
  Architecture: simplified


In [4]:
# Complete Logger Test and Integration
print("🧪 EM43 Logger Complete Test")
print("=" * 50)
print("NOTE: This test works with or without user registration.")
print("If not registered, most logging will be skipped (which is correct behavior).")
print("To test full functionality, run: python register.py first")
print("=" * 50)

# Step 1: Check user registration
print("\n1. 👤 User Registration Check")
if user_config_path.exists():
    with open(user_config_path, 'r') as f:
        user_data = json.load(f)
    
    print("✅ User already registered!")
    print(f"  Username: {user_data['username']}")
    print(f"  User ID: {user_data['user_id']}")
    print(f"  Created: {user_data['created_at']}")
    
    # Verify user with API
    try:
        response = requests.get(f"{api_endpoint}/user/{user_data['user_id']}/verify", timeout=20)
        if response.status_code == 200:
            print("✅ User verification successful!")
        else:
            print(f"❌ User verification failed: {response.status_code}")
    except Exception as e:
        print(f"❌ User verification error: {e}")
else:
    print("❌ User not registered!")
    print("Please run register.py first or use the manual registration below:")
    
    # Manual registration option
    def register_test_user(username):
        """Register a test user"""
        try:
            registration_data = {"username": username}
            response = requests.post(
                f"{api_endpoint}/register_user",
                json=registration_data,
                timeout=20,
                headers={'Content-Type': 'application/json'}
            )
            
            if response.status_code == 200:
                user_data = response.json()
                print("✅ Registration successful!")
                print(f"  User ID: {user_data['user_id']}")
                print(f"  Username: {user_data['username']}")
                
                # Save user config
                with open(user_config_path, 'w') as f:
                    json.dump(user_data, f, indent=2)
                
                print(f"✅ User config saved to {user_config_path}")
                return user_data
            else:
                print(f"❌ Registration failed: {response.status_code}")
                return None
        except Exception as e:
            print(f"❌ Registration error: {e}")
            return None
    
    # Uncomment and run the line below to register a new user for testing
    print("    To register a test user, uncomment and run:")
    print("    register_test_user(\"test_researcher_\" + datetime.now().strftime(\"%Y%m%d_%H%M%S\"))")
    print("    Or run: python register.py")

# Step 2: Initialize logger
print("\n2. 🚀 Logger Initialization")
logger = EM43Logger()

# Get logger status
status = logger.get_status()
print(f"  Enabled: {status['enabled']}")
print(f"  User registered: {status['user_registered']}")
print(f"  Config loaded: {status['config_loaded']}")
print(f"  API endpoint: {status['api_endpoint']}")

if status['user_registered']:
    print(f"  Username: {status['username']}")
    print(f"  User ID: {status['user_id']}")
    print(f"  Current run ID: {status['current_run_id']}")

# Test API connection
print("\n3. 🔍 API Connection Test")
connection_ok = logger.test_connection()
print(f"Connection status: {'✅ OK' if connection_ok else '❌ Failed'}")

# Step 3: Mode detection testing
print("\n4. 🎯 Mode Detection Test")

# Test 1-input task
task_1input = {
    'task_id': 1,
    'description': 'multiply by 2',
    'EDH': 's2,ep1,dp1,hf0.5ba',
    'num_inputs': 1
}

mode_1 = logger.detect_mode(task_1input)
print(f"  Task 1 (multiply by 2): {mode_1}")

# Test 2-input task
task_2input = {
    'task_id': 11,
    'description': 'addition of two inputs',
    'EDH': 's2,ep1,dp1,hf0.5ba',
    'num_inputs': 2
}

mode_2 = logger.detect_mode(task_2input)
print(f"  Task 11 (addition): {mode_2}")

# Test with description patterns
task_sum = {
    'task_id': 5,
    'description': 'sum two values',
    'EDH': 's2,ep1,dp1,hf0.5ba'
}

mode_sum = logger.detect_mode(task_sum)
print(f"  Task 5 (sum): {mode_sum}")

# Step 4: Manual logging test
print("\n5. 📝 Manual Logging Test")

test_task_config = {
    'task_id': 1,
    'description': 'multiply by 2',
    'EDH': 's2,ep1,dp1,hf0.5ba',
    'num_inputs': 1
}

# Log initial state
success_initial = logger.log_initial(
    task_config=test_task_config,
    initial_fitness=0.1,
    population_size=50,
    training_params={
        'mutation_rate': 0.02,
        'crossover_rate': 0.8,
        'selection_method': 'tournament'
    }
)

print(f"  Initial logging: {'✅ Success' if success_initial else '❌ Failed'}")

# Log checkpoint
success_checkpoint = logger.log_checkpoint(
    generation=10,
    best_fitness=0.75,
    avg_fitness=0.45,
    task_config=test_task_config,
    population_size=50,
    additional_data={'notes': 'Test checkpoint logging'}
)

print(f"  Checkpoint logging: {'✅ Success' if success_checkpoint else '❌ Failed'}")

# Log evaluation
success_eval = logger.log_evaluation(
    generation=20,
    best_fitness=0.85,
    avg_fitness=0.65,
    task_config=test_task_config,
    eval_fitness=0.92
)

# Check if evaluation logging is enabled
eval_enabled = logger.config['logging']['log_evaluation']
if eval_enabled:
    print(f"  Evaluation logging: {'✅ Success' if success_eval else '❌ Failed'}")
else:
    print(f"  Evaluation logging: ✅ Skipped (disabled in config)")

print("\n6. 🧬 EM43 Wrapper Integration Test")

try:
    # Create small test configuration overrides
    config_overrides = {
        'population': {
            'pop_size': {'value': 20},
            'generations': {'value': 5}
        },
        'model': {
            'mutation_rate': {'value': 0.02},
            'crossover_rate': {'value': 0.8}
        },
        'training': {
            'save_dir': {'value': './test_training_logs'},
            'random_seed': {'value': 42}
        }
    }
    
    # Initialize wrapper with task_id and config overrides
    wrapper = EM43Wrapper(task_id=1, config_overrides=config_overrides)
    
    print("✅ EM43 Wrapper initialized successfully")
    print(f"  Task ID: {wrapper.config['input_output']['task_id']['value']}")
    print(f"  Population size: {wrapper.config['population']['pop_size']['value']}")
    print(f"  Generations: {wrapper.config['population']['generations']['value']}")
    
    # Get task configuration for logging
    task_config = {
        'task_id': wrapper.config['input_output']['task_id']['value'],
        'description': 'multiply by 2',
        'EDH': 's2,ep1,dp1,hf0.5ba',
        'num_inputs': 1
    }
    
    # Log initial state
    print("  📊 Logging initial state...")
    logger.log_initial(
        task_config=task_config,
        initial_fitness=0.0,
        population_size=wrapper.config['population']['pop_size']['value'],
        training_params={
            'mutation_rate': wrapper.config['population']['mut_prog']['value'],
            'crossover_rate': 0.8,  # Not directly in config structure
            'selection_method': 'tournament',
            'num_generations': wrapper.config['population']['generations']['value']
        }
    )
    
    # Simulate training loop with logging
    print("  🏃 Running mini training with logging...")
    
    for gen in range(1, 4):  # Just 3 generations for testing
        # Simulate fitness improvement
        best_fitness = 0.1 + (gen * 0.2)
        avg_fitness = best_fitness * 0.6
        
        print(f"    Generation {gen}: best={best_fitness:.3f}, avg={avg_fitness:.3f}")
        
        # Log checkpoint
        logger.log_checkpoint(
            generation=gen,
            best_fitness=best_fitness,
            avg_fitness=avg_fitness,
            task_config=task_config,
            population_size=wrapper.config['population']['pop_size']['value'],
            additional_data={'generation_notes': f'Test generation {gen}'}
        )
    
    print("  ✅ Mini training with logging completed!")
    
except Exception as e:
    print(f"❌ EM43 Wrapper integration error: {e}")
    import traceback
    traceback.print_exc()

print("\n7. 🔍 Data Verification")

# Verify data was logged by checking the API
try:
    response = requests.get(f"{api_endpoint}/data", timeout=20)
    if response.status_code == 200:
        data_response = response.json()
        logged_data = data_response.get('data', [])
        
        print(f"✅ Retrieved {len(logged_data)} total logged entries")
        
        # Find our recent entries
        current_run_id = logger.get_run_id()
        our_entries = [entry for entry in logged_data 
                      if entry.get('data', {}).get('run_id') == current_run_id]
        
        print(f"✅ Found {len(our_entries)} entries from current run")
        
        # Display recent entries
        if our_entries:
            print("\n📊 Recent logged entries:")
            for i, entry in enumerate(our_entries[-5:]):  # Show last 5
                data = entry.get('data', {})
                print(f"  Entry {i+1}:")
                print(f"    Model: {data.get('model')}")
                print(f"    Mode: {data.get('mode')}")
                print(f"    Generation: {data.get('generation')}")
                print(f"    Checkpoint type: {data.get('checkpoint_type')}")
                print(f"    Best fitness: {data.get('best_fitness')}")
                print(f"    Timestamp: {entry.get('checkpoint_timestamp')}")
                print()
        
        # Create summary DataFrame
        if our_entries:
            df_data = []
            for entry in our_entries:
                data = entry.get('data', {})
                df_data.append({
                    'generation': data.get('generation', 0),
                    'checkpoint_type': data.get('checkpoint_type', 'unknown'),
                    'best_fitness': data.get('best_fitness', 0),
                    'avg_fitness': data.get('avg_fitness', 0),
                    'model': data.get('model', 'unknown'),
                    'mode': data.get('mode', 'unknown')
                })
            
            df = pd.DataFrame(df_data)
            print("📊 Summary of logged data:")
            print(df.to_string(index=False))
            
    else:
        print(f"❌ Failed to retrieve data: {response.status_code}")
        
except Exception as e:
    print(f"❌ Data verification error: {e}")

print("\n🎉 EM43 Logger Test Complete!")
print("=" * 50)

# Final summary
final_status = logger.get_status()
print(f"✅ Logger enabled: {final_status['enabled']}")
print(f"✅ User registered: {final_status['user_registered']}")
print(f"✅ Config loaded: {final_status['config_loaded']}")

if final_status['user_registered']:
    print(f"✅ Username: {final_status['username']}")
    print(f"✅ Current run ID: {final_status['current_run_id']}")

if final_status['user_registered']:
    print("\n🚀 System Status: READY FOR PRODUCTION!")
    print("\n📋 Next Steps:")
    print("1. Integrate logging into your training scripts")
    print("2. Use logger.log_initial() at the start of training")
    print("3. Use logger.log_checkpoint() during training")
    print("4. Use logger.log_evaluation() after evaluation")
    print("5. Check the API /data endpoint to view all logged data")
    print("\n🎯 Logger is ready for distributed research! 🧬")
else:
    print("\n🔧 System Status: NEEDS REGISTRATION!")
    print("\n📋 Next Steps:")
    print("1. Run: python register.py")
    print("2. Re-run this test to verify full functionality")
    print("3. Then integrate logging into your training scripts")
    print("\n💡 Logger will work but skip logging until registered.")


🧪 EM43 Logger Complete Test
NOTE: This test works with or without user registration.
If not registered, most logging will be skipped (which is correct behavior).
To test full functionality, run: python register.py first

1. 👤 User Registration Check
✅ User already registered!
  Username: giacomobocchese
  User ID: 2f6da8c3-cb7e-48fb-9ea0-b1cb5b1e515e
  Created: 2025-07-11T12:32:49.631860Z
✅ User verification successful!

2. 🚀 Logger Initialization
  Enabled: True
  User registered: True
  Config loaded: True
  API endpoint: https://jkk4nk8j9g.execute-api.us-east-1.amazonaws.com/api
  Username: giacomobocchese
  User ID: 2f6da8c3-cb7e-48fb-9ea0-b1cb5b1e515e
  Current run ID: None

3. 🔍 API Connection Test
✅ API health check passed
✅ User verification passed
Connection status: ✅ OK

4. 🎯 Mode Detection Test
  Task 1 (multiply by 2): 1input
  Task 11 (addition): 2input
  Task 5 (sum): 2input

5. 📝 Manual Logging Test
✅ Training data logged successfully [2025-07-11T12:37:40.717305Z]
  Init